<a target="_blank" href="https://colab.research.google.com/github/sergiopaniego/RAG_local_tutorial/blob/main/example_rag.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Exemple RAG simple avec Langchain, Ollama et un modèle LLM open source

Dans cet exemple, nous nous connectons d'abord à un LLM localement et faisons une demande au LLM qu'Ollama sert en utilisant LangChain. Après cela, nous générons notre application RAG à partir d'un fichier PDF et extrayons les détails de ce document.

# Installation des exigences

If an error is raised related to docarray, refer to this solution: https://stackoverflow.com/questions/76880224/error-using-using-docarrayinmemorysearch-in-langchain-could-not-import-docarray

In [8]:
!pip3 install langchain
!pip3 install langchain_pinecone
!pip3 install langchain[docarray]
!pip3 install docarray
!pip3 install pypdf

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

# Sélection du modèle LLM à utiliser

In [9]:
#MODEL = "gpt-3.5-turbo"
#MODEL = "mixtral:8x7b"
#MODEL = "gemma:7b"
#MODEL = "llama2"
MODEL = "llama3" # https://ollama.com/library/llama3

# Nous instancions le modèle LLM et le modèle Embedding

In [10]:
pip install langchain-community


[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings

model = Ollama(model=MODEL)
embeddings = OllamaEmbeddings(model=MODEL)

model.invoke("Donnez-moi une citation inspirante")

'Voici une citation inspirante de Nelson Mandela :\n\n"Les victoires sont souvent les résultats de la détermination, du courage et de l\'esprit d\'équipe. Les défaites sont également des opportunités pour apprendre, grandir et se relever."\n\n(J\'ai traduit pour vous !)\n\nJ\'espère que cela vous inspire et vous donne confiance !'

In [5]:
model.invoke("que vaut 2+2?")

'Une question classique !\n\nLa réponse est : 4'

## En utilisant un analyseur fourni par LangChain, nous pouvons transformer la sortie LLM en quelque chose de plus adapté à la lecture

In [15]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
response_from_model = model.invoke("Donnez-moi une citation inspirante")
parsed_response = parser.parse(response_from_model)
print(parsed_response)

Voici une citation inspirante :

« Il n'y a pas d'échecs, seul les retours sont possibles. Les vrais leaders ne se laissent jamais définir par leurs erreurs, mais par leur capacité à rebondir après celles-ci. »

— Nelson Mandela


# Nous générons le modèle de conversation avec le modèle d'apprentissage des langues basé sur des instructions

Nous pouvons créer un modèle pour structurer efficacement la conversation.

Ce modèle nous permet de fournir un contexte général au modèle d'apprentissage des langues (LLM), qui sera utilisé pour chaque invite. Cela garantit que le modèle a une compréhension de fond cohérente pour toutes les interactions.

De plus, nous pouvons inclure un contexte spécifique pertinent pour l'invite particulière. Cela aide le modèle à comprendre le scénario ou le sujet immédiat avant d'aborder la question réelle. En suivant ce contexte spécifique, nous présentons ensuite la question réelle à laquelle nous voulons que le modèle réponde.

En utilisant cette approche, nous améliorons la capacité du modèle à générer des réponses précises et pertinentes basées à la fois sur les contextes généraux et spécifiques fournis.

In [12]:
from langchain.prompts import PromptTemplate

template = """
Répondez à la question en fonction du contexte ci-dessous. Si vous ne pouvez pas
répondre à la question, répondez par « Je ne sais pas ».

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt.format(context="Voici un peu de contexte", question="Voici une question")

'\nRépondez à la question en fonction du contexte ci-dessous. Si vous ne pouvez pas\nrépondre à la question, répondez par « Je ne sais pas ».\n\nContext: Voici un peu de contexte\n\nQuestion: Voici une question\n'

Le modèle peut répondre aux questions en fonction du contexte :

In [16]:
formatted_prompt = prompt.format(context="Mes parents m'ont nommé Falissou", question="Quel est mon nom?")
response_from_model = model.invoke(formatted_prompt)
parsed_response = parser.parse(response_from_model)
print(parsed_response)

Ton nom est Falissou !


Mais il ne peut pas répondre à ce qui n'est pas fourni comme contexte :

In [17]:
formatted_prompt = prompt.format(context="Mes parents m'ont nommé Falissou", question="Quel est mon age?")
response_from_model = model.invoke(formatted_prompt)
parsed_response = parser.parse(response_from_model)
print(parsed_response)

Je ne sais pas.


Même des informations déjà connues !

In [23]:
formatted_prompt = prompt.format(context="Mes parents m'ont nommé Falissou", question="Que vaut 2+2?")
response_from_model = model.invoke(formatted_prompt)
parsed_response = parser.parse(response_from_model)
print(parsed_response)

Je ne sais pas.


# Chargez un exemple PDF pour effectuer une Génération Augmentée de Récupération (RAG)

Pour l'exemple, vous pouvez sélectionner votre propre PDF.

In [18]:
pip install pypdf


[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("./Lettre_motivation.pdf")
pages = loader.load_and_split()
#pages = loader.load()
pages

[Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content="Falissou AMBOUSSIDI                         Dakar, le 10 janvier 2025 \n77 397 82 23 \namboussidifalissou@gmail.com \n \n \n    Au \n Directeur général. \nAgence nationale de la Statistique et de la \nDémographie.  \n \n \nObjet : Candidature pour un stage au profil de Data Scientiste \nMadame, Monsieur, \nEtudiant en fin de formation en Master Intelligence Artificielle et Big Data à l’École Supérieure Polytechnique de \nl’Université Cheik Anta Diop de Dakar, je me permets de vous adresser ma candidature pour le profil de Data Scientiste. \nPassionné par les technologies innovantes et la transformation digitale, je souhaite mettre mes compétences en œuvre \nau service des projets menés par l’Agence nationale de la Statistique et de la Démographie. \nAu cours de ma formation, j’ai acquis des compétences en  mathématique puis  en intelligence artificielle, machine \nlearning, deep learning et Big Data. J’a

In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text_documents = text_splitter.split_documents(pages)[:5]

pages

[Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content="Falissou AMBOUSSIDI                         Dakar, le 10 janvier 2025 \n77 397 82 23 \namboussidifalissou@gmail.com \n \n \n    Au \n Directeur général. \nAgence nationale de la Statistique et de la \nDémographie.  \n \n \nObjet : Candidature pour un stage au profil de Data Scientiste \nMadame, Monsieur, \nEtudiant en fin de formation en Master Intelligence Artificielle et Big Data à l’École Supérieure Polytechnique de \nl’Université Cheik Anta Diop de Dakar, je me permets de vous adresser ma candidature pour le profil de Data Scientiste. \nPassionné par les technologies innovantes et la transformation digitale, je souhaite mettre mes compétences en œuvre \nau service des projets menés par l’Agence nationale de la Statistique et de la Démographie. \nAu cours de ma formation, j’ai acquis des compétences en  mathématique puis  en intelligence artificielle, machine \nlearning, deep learning et Big Data. J’a

# Stockage du PDF dans un espace vectoriel.

Extrait de la documentation Langchain :

Le temps d'exécution du bloc suivant dépend de la complexité et de la longitude du PDF fourni. Essayez de le garder petit et simple pour l'exemple.

In [21]:
pip install docarray


[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(text_documents, embedding=embeddings)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


# Créer un récupérateur de vecteurs similaires à utiliser comme contexte

In [23]:
retriever = vectorstore.as_retriever()
retriever.invoke("intelligence artificielle")

[Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content='Démographie.  \n \n \nObjet : Candidature pour un stage au profil de Data Scientiste'),
 Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content='Madame, Monsieur,'),
 Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content='77 397 82 23 \namboussidifalissou@gmail.com \n \n \n    Au \n Directeur général.'),
 Document(metadata={'source': './Lettre_motivation.pdf', 'page': 0}, page_content='Falissou AMBOUSSIDI                         Dakar, le 10 janvier 2025 \n77 397 82 23')]

# Générer une conversation avec le document pour extraire les détails

In [24]:
# En supposant que Retriever est une instance d'une classe Retriever et dispose d'une méthode pour récupérer le contexte
retrieved_context = retriever.invoke("intelligence artificielle")

In [29]:
questions = [
    "Quel est l'object de la lettre?",
    "Quelle sont les competences techniques cités",
    "L'auteur etudie dans quelle ecole?"
]

for question in questions:
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = model.invoke(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Question: {question}")
    print(f"Answer: {parsed_response}")
    print()

Question: Quel est l'object de la lettre?
Answer: Selon le contexte, l'objet de la lettre est "Candidature pour un stage au profil de Data Scientiste".

Question: Quelle sont les competences techniques cités
Answer: Je ne sais pas. Le contexte ne présente pas de compétences techniques citées dans le document. Il s'agit plutôt d'une lettre de motivation pour un stage en tant que Data Scientiste, avec des informations personnelles et d'adresse.

Question: L'auteur etudie dans quelle ecole?
Answer: Je ne sais pas. Le contexte ne fournit pas d'information sur l'école où étudie l'auteur.



# Boucle pour poser et répondre à des questions en continu

In [ ]:
while True:
    print("Ecrire 'sortir' ou 'quitter' pour quitter la discussion")
    question = input('Question: ')
    print(f"Question: {question}")
    if question.lower() in ["sortir", "quitter"]:
        print("Fin de la discussion. Aurevoir!")
        break
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = model.invoke(formatted_prompt)
    parsed_response = parser.parse(response_from_model)
    print(f"Answer: {parsed_response}")
    print()

Ecrire 'sortir' ou 'quitter' pour quitter la discussion


# Évaluation des performances du LLM :

Pour évaluer les performances du modèle de langage, nous utilisons des métriques standard, notamment la précision, le rappel, et la F1-score.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Réponses attendues pour les questions
expected_answers = [
    "L'objet de la lettre est une Candidature pour un stage au profil de Data Scientiste.",
    "Les compétences techniques de l'auteur incluent Python, TensorFlow, et Pandas, ainsi que des frameworks de Big Data comme Apache Spark et Hadoop.",
    "L'auteur étudie à l'École Supérieure Polytechnique de Dakar."
]

# Évaluer les réponses générées par le modèle
generated_answers = []

for question in questions:
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = model.invoke(formatted_prompt)
    parsed_response = parser.parse(response_from_model)
    generated_answers.append(parsed_response)

    print(f"Question: {question}")
    print(f"Generated Answer: {parsed_response}")
    print(f"Expected Answer: {expected_answers[questions.index(question)]}")
    print()

# Calcul des métriques
# Convertir les réponses en vecteurs binaires pour précision, rappel, etc.
binary_generated = [1 if answer == expected else 0 for answer, expected in zip(generated_answers, expected_answers)]

accuracy = accuracy_score([1] * len(expected_answers), binary_generated)
precision = precision_score([1] * len(expected_answers), binary_generated, zero_division=0)
recall = recall_score([1] * len(expected_answers), binary_generated, zero_division=0)
f1 = f1_score([1] * len(expected_answers), binary_generated, zero_division=0)

print("Évaluation des performances :")
print(f"Accuracy : {accuracy:.2f}")
print(f"Précision : {precision:.2f}")
print(f"Rappel : {recall:.2f}")
print(f"F1-score : {f1:.2f}")


Question: Quel est l'object de la lettre?
Generated Answer: Selon le contexte, l'objet de la lettre est : "Candidature pour un stage au profil de Data Scientiste".
Expected Answer: L'objet de la lettre est de postuler pour un stage.

Question: Quelle sont les competence de l'auteur?
Generated Answer: Je ne sais pas. Le contexte ne fournit pas d'informations sur les compétences de l'auteur, mais plutôt des informations personnelles (adresse e-mail, numéro de téléphone) et une lettre de motivation pour un stage en tant que Data Scientiste.
Expected Answer: Les compétences de l'auteur incluent l'apprentissage automatique et l'intelligence artificielle.

Question: Ou etudie l'auteur de la lettre?
Generated Answer: Dakar.
Expected Answer: L'auteur étudie à l'École Supérieure Polytechnique de Dakar.

Évaluation des performances :
Accuracy : 0.00
Précision : 0.00
Rappel : 0.00
F1-score : 0.00


# Analyse de la robustesse :

La robustesse peut être testée en introduisant des variations dans les données d'entrée (par exemple, paraphrases des questions, contextes incomplets ou bruités) et en mesurant la cohérence des réponses générées.

In [27]:
# Variations des questions pour évaluer la robustesse
variant_questions = [
    "Quel est l'object de la lettre?",
    "Quelle sont les competences techniques cités",
    "L'auteur etudie dans quelle ecole?"
]

robustness_results = []

for question in variant_questions:
    formatted_prompt = prompt.format(context=retrieved_context, question=question)
    response_from_model = model.invoke(formatted_prompt)
    parsed_response = parser.parse(response_from_model)

    print(f"Variant Question: {question}")
    print(f"Generated Answer: {parsed_response}")

    # Calcul de la similarité avec les réponses attendues
    similarity = 1 if parsed_response in expected_answers else 0
    robustness_results.append(similarity)

robustness_score = sum(robustness_results) / len(robustness_results)

print("\nAnalyse de la robustesse :")
print(f"Score de robustesse : {robustness_score:.2f}")


Variant Question: Quel est le sujet de cette lettre ?
Generated Answer: En fonction du contexte, je réponds : Candidature pour un stage au profil de Data Scientiste.
Variant Question: Quelles sont les compétences mentionnées dans le document ?
Generated Answer: Je ne sais pas.
Variant Question: Où l'auteur poursuit-il ses études ?
Generated Answer: Je ne sais pas. Il n'y a pas d'information sur l'emplacement de l'université ou de l'établissement où l'auteur poursuit ses études dans le contexte fourni.

Analyse de la robustesse :
Score de robustesse : 0.00
